# Querying a Fabric Data Agent with the Responses API

This notebook guides you how to use `FabricOpenAIResponses` client that is based on the OpenAI **Responses API**.

## 1. Install the SDK

You can use the same package as before. It ships both `FabricOpenAI` client and
`FabricOpenAIResponses` client.

In [ ]:
# Install (or upgrade) the Fabric Data Agent SDK into the notebook kernel.
%pip install -U fabric-data-agent-sdk

## 2. Point at a data agent and create the Responses client

Unlike the Assistants API, there is **no assistant and no thread to create**. You
build one client and call it directly.

`FabricOpenAIResponses` takes the agent name, the workspace, and a stage
(`"sandbox"` for the draft version, `"production"` for the published one).

In [ ]:
import sempy.fabric as fabric
from fabric.dataagent.client import FabricOpenAIResponses

# Replace these with your own data agent and workspace names.
data_agent_name = "<your-data-agent-name>"
workspace_name = "<your-workspace-name>"

# Resolve the workspace name to an ID (skip if you already have the ID).
workspace_id = fabric.resolve_workspace_id(workspace_name) if workspace_name else None

# Create the Responses client. There is nothing else to set up before asking.
fabric_client = FabricOpenAIResponses(
    artifact_name=data_agent_name,          # which data agent to query
    workspace_name=workspace_id or workspace_name,  # where it lives
    ai_skill_stage="sandbox",               # 'sandbox' (draft) or 'published'
)

# Quick sanity check of what the client is bound to.
print(f"api_type: {fabric_client.api_type}")           # 'responses'
print(f"default_model: {fabric_client.default_model}")  # the model the SDK injects
print(f"workspace: {fabric_client.workspace_name}")
print(f"artifact: {fabric_client.artifact_name}")

## 3. Helper functions

These helpers keep the querying cells short and read the answer and the
intermediate steps out of a response no matter which shape it arrives in.

Each helper is explained in its comments.

In [ ]:
import time

# Statuses that mean the response is finished (successfully or not).
TERMINAL_STATUSES = {"completed", "failed", "incomplete", "cancelled"}


def get_field(value, name, default=None):
    # Read a field whether 'value' is a dict or an SDK object.
    if isinstance(value, dict):
        return value.get(name, default)   # dict access
    return getattr(value, name, default)  # attribute access


def response_output_items(response):
    # The answer, tool calls, and code all live in the response's 'output' list.
    return list(get_field(response, "output", []) or [])


def response_status(response):
    # Normalize the status to lowercase for comparison against TERMINAL_STATUSES.
    status = get_field(response, "status", None)
    return str(status).lower() if status else ""


def get_response_id(response):
    # Every response carries an id; we use it to poll and to chain follow-ups.
    return get_field(response, "id", None)


def wait_for_response(response, timeout_seconds=600, poll_interval_seconds=2):
    # Poll the response by id until it reaches a terminal status.
    start_time = time.monotonic()
    while response_status(response) not in TERMINAL_STATUSES:
        # Give up if it takes too long.
        if time.monotonic() - start_time > timeout_seconds:
            raise TimeoutError(f"Timed out waiting for response {get_response_id(response) or ''}")
        response_id = get_response_id(response)
        if not response_id:
            raise ValueError(f"Response has no id to poll. Raw response: {response!r}")
        time.sleep(poll_interval_seconds)                         # wait between checks
        response = fabric_client.responses.retrieve(response_id)  # re-read by id
    return response


def extract_response_text(response):
    # Pull the answer text out of the 'message' output items.
    parts = []
    for item in response_output_items(response):
        if get_field(item, "type") != "message":   # only message items hold text
            continue
        for content in get_field(item, "content", []) or []:
            text = get_field(content, "text")       # the actual text value
            if text:
                parts.append(str(text))
    # Join message text, or fall back to the convenience 'output_text' field.
    return "\n\n".join(parts) or str(get_field(response, "output_text", "") or "")

## 4. Ask one question

This replaces four Assistants calls (`messages.create` + `runs.create`, preceded by
`assistants.create` + `threads.create`) with a single `responses.create`.

Note there is **no `model` argument**. The SDK injects the default Responses model.

In [ ]:
first_question = "Give me the month with the most public holidays."

# One call sends the question. No assistant, no thread, no separate run.
first_response = fabric_client.responses.create(input=first_question)

# Wait until the response reaches a terminal status.
first_response = wait_for_response(first_response)

# Read the answer text.
first_answer = extract_response_text(first_response)
print("status:", response_status(first_response))
print("answer:", first_answer)

## 5. Continue the conversation with `previous_response_id`

In the Assistants API you kept posting to the same thread. Here you pass the id of
the previous response, and the service keeps the context. No thread required.

In [ ]:
# Take the id of the answer we just received.
previous_response_id = get_response_id(first_response)

# Ask a follow-up, chaining from the previous response.
conversation_response = fabric_client.responses.create(
    input="Which month follows the month from your previous answer?",
    previous_response_id=previous_response_id,  # this is what carries the context
)
conversation_response = wait_for_response(conversation_response)

print("answer:", extract_response_text(conversation_response))

## 6. Continue the conversation with a `conversation` id

The other option is to create a conversation once and pass its id on every turn.
Use this when you want a single id that groups a whole exchange, instead of chaining
response to response.

In [ ]:
# Create a conversation object once.
conversation = fabric_client.conversations.create()
conversation_id = get_field(conversation, "id")

# First turn, tagged with the conversation id.
turn_one = fabric_client.responses.create(
    input="According to the public holidays table, which month has the most holidays?",
    conversation=conversation_id,
)
turn_one = wait_for_response(turn_one)
print("turn 1:", extract_response_text(turn_one))

# Second turn on the same conversation id keeps the context.
turn_two = fabric_client.responses.create(
    input="Which month follows the month from your previous answer?",
    conversation=conversation_id,
)
turn_two = wait_for_response(turn_two)
print("turn 2:", extract_response_text(turn_two))

## 7. Stream the answer

Streaming is available in the Assistants API as well. With the Responses API,
you open a stream and read events as they arrive. Text arrives as
`response.output_text.delta` events, and other events carry tool calls and the
final response.

For table or row requests, the useful result may come back as output items
rather than as streamed text, so do not rely on the printed text alone.

In [ ]:
streamed_text = ""

# Open a stream. It is a context manager, so use 'with'. We chain from an earlier
# response so the stream continues the same conversation.
with fabric_client.responses.stream(
    input="Give me the first 10 rows from the public holidays table.",
    previous_response_id=get_response_id(conversation_response),
) as stream:
    for event in stream:
        # Print text pieces as they arrive.
        if get_field(event, "type") == "response.output_text.delta":
            delta = str(get_field(event, "delta", ""))
            print(delta, end="")
            streamed_text += delta

print("\n\n(done streaming)")

## 8. Inspect the intermediate steps

In the Assistants API you called `runs.steps.list`. Here the same information is in
the response's output items: `function_call` (a tool call), `function_call_output`
(its result), and `code_interpreter_call` (executed code).

In [ ]:
import pandas as pd

# Send a request that uses the data source, so the response includes tool calls.
steps_response = fabric_client.responses.create(
    input="Give me the first 10 rows from the public holidays table.",
)
steps_response = wait_for_response(steps_response)

# Turn the tool-related output items into a small table for readability.
rows = []
for item in response_output_items(steps_response):
    item_type = get_field(item, "type")
    if item_type == "function_call":
        # A tool call: the tool name and the arguments it was called with.
        rows.append({"type": item_type, "name": get_field(item, "name"),
                     "arguments": get_field(item, "arguments"), "output": None})
    elif item_type == "function_call_output":
        # The result of a tool call.
        rows.append({"type": item_type, "name": get_field(item, "name"),
                     "arguments": None, "output": get_field(item, "output")})
    elif item_type == "code_interpreter_call":
        # Code that was executed, and its output.
        rows.append({"type": item_type, "name": item_type,
                     "arguments": get_field(item, "code") or get_field(item, "input"),
                     "output": get_field(item, "output")})

display(pd.DataFrame(rows))

## 9. Evaluation on the Responses path

`evaluate_data_agent` still defaults to the Assistants API. To run the same
evaluation against the Responses API, pass `client_class=FabricOpenAIResponses`.

Results land in two tables: `<table_name>` for row-level results and
`<table_name>_steps` for the per-step detail (function calls and tool outputs).

In [ ]:
import pandas as pd
from fabric.dataagent.evaluation import evaluate_data_agent

# A tiny ground-truth set: question plus the answer you expect.
eval_df = pd.DataFrame(
    [
        {
            "question": "According to the public holidays table, which month is New Years in?",
            "expected_answer": "January",
        }
    ]
)

# Run the evaluation against the Responses API by passing client_class.
evaluation_id = evaluate_data_agent(
    eval_df,
    data_agent_name=data_agent_name,
    workspace_name=workspace_id or workspace_name,
    table_name="responses_api_evaluation_output",
    data_agent_stage="sandbox",
    max_workers=1,
    client_class=FabricOpenAIResponses,  # <- the one line that selects the Responses path
)

print("evaluation_id:", evaluation_id)

## What changed, in one screen

- `FabricOpenAI` became `FabricOpenAIResponses`.
- No more `assistants.create` or `threads.create`.
- `messages.create` + `runs.create` became one `responses.create(input=...)`.
- Run-status polling became response-status polling (`responses.retrieve`).
- Reading `messages.list` / `content[0].text.value` became reading `output` items.
- Thread reuse became `previous_response_id` or a `conversation` id.
- `runs.steps.list` became `function_call` / `function_call_output` items.
- No `threads.delete` to call.

See `responses-api-migration-guide.md` for the full checklist.